In [35]:
!git clone https://github.com/AdityaD16/Quantum-Circuit-Simulator-DMRG.git
%cd Quantum-Circuit-Simulator-DMRG

Cloning into 'Quantum-Circuit-Simulator-DMRG'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 53 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 46.77 KiB | 870.00 KiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/Quantum-Circuit-Simulator-DMRG


In [36]:
!ls

LICENSE  main.py  README.md  requirements.txt  src


In [ ]:
!pip install -r requirements.txt

Definir uertas cuánticas como tensores: tensor network / MPS manual

In [39]:
import numpy as np
from scipy.linalg import expm

def hadamard_gate():
    """Returns the Hadamard gate as a 2x2 tensor."""
    return (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]])

def cnot_gate():
    """Returns the CNOT gate as a 2x2x2x2 tensor."""
    return np.array([[[[1, 0], [0, 0]], [[0, 0], [0, 1]]], [[[0, 0], [0, 1]], [[1, 0], [0, 0]]]])

def random_gate():
    """Returns a random 2-qubit unitary gate as a 2x2x2x2 tensor."""
    A = np.random.randn(4, 4) + 1j * np.random.randn(4, 4)
    Q, _ = np.linalg.qr(A)
    return Q.reshape(2, 2, 2, 2)

def random_isometry(rows, cols):
    """Returns a random isometry matrix reshaped as a tensor."""
    A = np.random.randn(rows, cols) + 1j * np.random.randn(rows, cols)
    Q, _ = np.linalg.qr(A)
    if cols > rows:
        Q_t, _ = np.linalg.qr(A.T)
        Q = Q_t.T
    return Q

def ZZ_QAOA(gamma):
    """Returns the ZZ interaction gate for QAOA with parameter gamma."""
    A = np.diag([np.exp(-0.5j * gamma), np.exp(0.5j * gamma), np.exp(0.5j * gamma), np.exp(-0.5j * gamma)])
    return A.reshape(2, 2, 2, 2)

def X_QAOA(beta):
    """Returns the X rotation gate for QAOA with parameter beta."""
    X_gate = np.array([[0, 1], [1, 0]])
    return expm(-0.5j * beta * X_gate)

Construir estados cuánticos y circuitos como redes tensoriales (MPS / árboles)incluyendo puertas tipo QAOA y evolución cuántica

cada circuito = grafo de tensores


In [43]:
# src/utils/circuit_gen.py

import numpy as np
import copy
from src.network import Network
from src.node import Node
from src.utils.quantum_gates import random_isometry, random_gate

def D_tree(network_structure, D_max): #Returns the bond dimension list for each layer of tree
    return [min(2**x, D_max) for x in network_structure][:-1]#controla el bond dimension en árboles tensoriales esto limita entanglement

def D_mps(no_qubits,D_max): # Returns the bond dimension list for MPS, simula estructura realista de entanglement en cadena 1D
    Dmps_ = [min(D_max, 2**(i+1)) for i in range(no_qubits // 2)]
    if no_qubits % 2 == 0:
        Dmps = Dmps_ + Dmps_[::-1][1:]
    else:
        Dmps = Dmps_ + Dmps_[::-1]
    return Dmps

def Tree(D,node_name,TR,initial,no_nodes): #construye un tensor network tipo árbol
    N = Network()
    N.rank_all = TR
    N.Layers = len(D)
    assert node_name == "Ket" or node_name == "Bra", "Wrong Name"
    if initial == "zero":
        for i in range(len(D)):
            for j in range(no_nodes[i]):
                if i==0:
                    rank = no_nodes[i+1]
                    tensor = np.zeros(([D[-1-i]]*rank))
                    tensor.flat[0] = 1
                    A = Node(node_name+f"{i}{j}",rank,[D[-1-i]]*rank,tensor)
                    N.add_node(A)
                else:

                    rank = no_nodes[i+1]//no_nodes[i] + 1

                    pi = i-1
                    pj = j//(no_nodes[i]//no_nodes[i-1])
                    if pi == 0:
                        idx = j
                    else:
                        idx = j+1 - (no_nodes[i]//no_nodes[i-1])*pj
                    dimensions = [D[-i]] + [D[-i-1]]*(rank-1)
                    tensor = np.zeros((dimensions))
                    tensor.flat[0] = 1
                    A = Node(node_name+f"{i}{j}",rank,dimensions,tensor)
                    N.add_node(A)
                    # print(pi,pj,idx)
                    # N.print_network()
                    N.add_to_node(node_name+f"{pi}{pj}",node_name+f"{i}{j}",idx,0)

    if initial == "random":
        for i in range(len(D)):
            for j in range(no_nodes[i]):
                if i==0:
                    rank = no_nodes[i+1]
                    dimensions = D[-i-1]**rank
                    A = Node(node_name+f"{i}{j}",rank,[D[-1-i]]*rank,(np.random.randn(dimensions) + 1j * np.random.randn(dimensions)).reshape([D[-i-1]]*rank))
                    N.add_node(A)
                else:
                    rank = no_nodes[i+1]//no_nodes[i] + 1
                    pi = i-1
                    pj = j//(no_nodes[i]//no_nodes[i-1])
                    if pi == 0:
                        idx = j
                    else:
                        idx = j +1 - (no_nodes[i]//no_nodes[i-1])*pj
                    dimensions = [D[-i]] + [D[-i-1]]*(rank-1)
                    tensor = random_isometry(D[-i],D[-i-1]**(rank-1))
                    # print(D[-i-1]*D[-i-1])
                    A = Node(node_name+f"{i}{j}",rank,dimensions,tensor.reshape(dimensions))
                    N.add_node(A)
                    N.add_to_node(node_name+f"{pi}{pj}",node_name+f"{i}{j}",idx,0)
    return N

def MPS(D,node_name,TR,initial): #construye Matrix Product State 1D

    N = Network()
    N.rank_all = TR
    N.Layers = 1
    assert node_name == "Ket" or node_name == "Bra", "Wrong Name"
    if initial == "random":
        for i in range(len(D)+1):
            if i == 0:
                A = Node(node_name+f"{i}",2,[D[i],2],np.random.randn(D[i],2) + 1j * np.random.randn(D[i],2))
                N.add_node(A)
            elif i == len(D):
                tensor = random_isometry(D[i-1],2)
                A = Node(node_name+f"{i}",2,[D[i-1],2],tensor.reshape(D[i-1],2))
                N.add_node(A)
                idx = 0 if (i-1 == 0) else 2
                N.add_to_node(node_name+f"{i-1}",node_name+f"{i}",idx,0)
            else:
                tensor = random_isometry(D[i-1],2*D[i])
                # print(tensor.shape)
                A = Node(node_name+f"{i}",3,[D[i-1],2,D[i]],tensor.reshape(D[i-1],2,D[i]))
                N.add_node(A)
                idx = 0 if (i-1 == 0) else 2
                N.add_to_node(node_name+f"{i-1}",node_name+f"{i}",idx,0)
                # N.print_network()

    if initial == "zero":
        for i in range(len(D)+1):
            zero_2 = np.zeros((1,2))
            zero_2[0,0] = 1
            zero_3 = np.zeros((1,2,1))
            zero_3[0,0,0] = 1
            if i == 0:
                A = Node(node_name+f"{i}",2,[D[i],2],zero_2)
                N.add_node(A)
            elif i == len(D):
                A = Node(node_name+f"{i}",2,[D[i-1],2],zero_2)
                N.add_node(A)
                idx = 0 if (i-1 == 0) else 2
                N.add_to_node(node_name+f"{i-1}",node_name+f"{i}",idx,0)
            else:
                A = Node(node_name+f"{i}",3,[D[i-1],2,D[i]],zero_3)
                N.add_node(A)
                idx = 0 if (i-1 == 0) else 2
                N.add_to_node(node_name+f"{i-1}",node_name+f"{i}",idx,0)
    return N

def circuit_from_edge_list(TR,edge_list,no_qubits,single_qubit_gate_ket,single_qubit_gate_bra,circuit_type,two_qubit_gate=None): #convierte un circuito en red tensorial (QAOA en formato tensorial)

    new_network = Network()
    new_network.rank_all = TR
    current_gates = []

    for i in range(no_qubits):
        A = Node(f"1G_K{i}",2,[2,2],single_qubit_gate_ket)
        current_gates.append(f"1G_K{i}")
        new_network.add_node(A)

    assert circuit_type == "random" or circuit_type == "qaoa", "Wrong circuit type"

    for i in edge_list:
        x,y = i

        # if x<y:
        #     name_2q = f"2G_{x}_{y}"
        # else:
        #     name_2q = f"2G_{y}_{x}"
        name_2q = f"2G_{x}_{y}"
        # print(i,current_gates[x],name_2q)
        if circuit_type == "random":
            A = Node(name_2q,4,[2,2,2,2],random_gate())
        else:
            assert two_qubit_gate is not None, "Wrong two qubit gate for QAOA"
            A = Node(name_2q,4,[2,2,2,2],two_qubit_gate)
        new_network.add_node(A)

        if current_gates[x][0] == "1":
            new_network.add_to_node(current_gates[x],name_2q,1,0)
            current_gates[x] = name_2q
        else:
            indices = [dash for dash, val in enumerate(current_gates[x]) if val == '_']
            if current_gates[x][indices[0]+1:indices[1]] == str(x):
                new_network.add_to_node(current_gates[x],name_2q,2,0)
                current_gates[x] = name_2q
            else:
                new_network.add_to_node(current_gates[x],name_2q,3,0)
                current_gates[x] = name_2q

        if current_gates[y][0] == "1":
            new_network.add_to_node(current_gates[y],name_2q,1,1)
            current_gates[y] = name_2q
        else:
            indices = [dash for dash, val in enumerate(current_gates[y]) if val == '_']
            if current_gates[y][indices[0]+1:indices[1]] == str(y):
                new_network.add_to_node(current_gates[y],name_2q,2,1)
                current_gates[y] = name_2q
            else:
                new_network.add_to_node(current_gates[y],name_2q,3,1)
                current_gates[y] = name_2q



    for i in range(no_qubits):
        A = Node(f"1G_B{i}",2,[2,2],single_qubit_gate_bra)
        new_network.add_node(A)
        node = new_network.nodes[current_gates[i]]
        # idx = node.subscript.index(node.open_legs[0])
        if current_gates[i][0] =="1":
            new_network.add_to_node(f"1G_B{i}",current_gates[i],0,1)
        else:
            indices = [dash for dash, val in enumerate(current_gates[i]) if val == '_']
            if str(i) == current_gates[i][indices[0]+1:indices[1]]:
                new_network.add_to_node(f"1G_B{i}",current_gates[i],0,2)
            else:
                new_network.add_to_node(f"1G_B{i}",current_gates[i],0,3)
    return new_network

def full_network_from_edge_list(Ket,Circuit,Bra,no_qubits): #esto permite calcular:⟨ψ|U|ψ⟩, energías

    new_network = Network()
    for name, node in Ket.nodes.items():
        new_network.nodes[name] = copy.deepcopy(node)

    for name, node in Circuit.nodes.items():
        new_network.nodes[name] = copy.deepcopy(node)

    for name, node in Bra.nodes.items():
        new_network.nodes[name] = copy.deepcopy(node)

    # print(new_network.leaf_nodes)

    new_network.leaf_nodes = copy.deepcopy(Ket.leaf_nodes)
    # print(new_network.leaf_nodes)
    # new_network.print_network()
    for i in range(no_qubits):
        node = new_network.nodes[new_network.leaf_nodes[0]]
        idx = node.subscript.index(node.open_legs[0])
        new_network.add_to_node(f"1G_K{i}",new_network.leaf_nodes[0],0,idx) # Leaf node gets removed


    new_network.leaf_nodes = copy.deepcopy(Bra.leaf_nodes)
    for i in range(no_qubits):
        node = new_network.nodes[new_network.leaf_nodes[0]]
        idx = node.subscript.index(node.open_legs[0])
        new_network.add_to_node(f"1G_B{i}",new_network.leaf_nodes[0],1,idx)

    return new_network

def ket_network_from_edge_list(Ket,Circuit,no_qubits):

    new_network = Network()
    for name, node in Ket.nodes.items():
        new_network.nodes[name] = copy.deepcopy(node)

    for name, node in Circuit.nodes.items():
        new_network.nodes[name] = copy.deepcopy(node)


    new_network.leaf_nodes = copy.deepcopy(Ket.leaf_nodes)
    for i in range(no_qubits):
        node = new_network.nodes[new_network.leaf_nodes[0]]
        idx = node.subscript.index(node.open_legs[0])
        new_network.add_to_node(f"1G_K{i}",new_network.leaf_nodes[0],0,idx)


    return new_network

In [44]:
import networkx as nx

def nearest_neighbour_edge_list(no_qubits):
    """Generates a linear chain (1D nearest-neighbor) edge list."""
    G = nx.Graph()
    G.add_nodes_from(range(no_qubits))
    for i in range(no_qubits-1):
        G.add_edge(i,i+1)
    return G
def generate_3_regular_graph(num_qbts, seed=None):
    """Generates a random 3-regular graph."""
    if num_qbts % 2 != 0 or num_qbts <= 3:
        raise ValueError("A 3-regular graph requires an even number of vertices > 3.")
    G = nx.random_regular_graph(3, num_qbts, seed=seed)
    return G

def tree_like_edge_list(network_structure):
    """Generates a tree-like edge list based on the network hierarchy."""
    G = nx.Graph()
    num_nodes = network_structure[-1]
    G.add_nodes_from(range(num_nodes))
    for l in range(len(network_structure)-1):
        cluster_size = network_structure[-1]//network_structure[-2-l]
        clusters = []

        for i in range(0, num_nodes, cluster_size):
            cluster_nodes = list(range(i, i + cluster_size,network_structure[-1]//network_structure[-1-l]))
            clusters.append(cluster_nodes)
            # Fully connect nodes within each cluster
            for j in cluster_nodes:
                for k in cluster_nodes:
                    if j!=k:
                        G.add_edge(j,k)
    return G

def remove_edge_and_get_endpoints(G):
    for u, v in list(G.edges()):
        if G.degree[u] == 3 and G.degree[v] == 3:
            G.remove_edge(u, v)
            return (u, v)
    raise ValueError("No suitable edge found to remove.")

def construct_bridged_graph(num_units, vertices_per_unit):
    """Constructs a graph from smaller, highly-connected units linked by bridges."""
    units = []
    removed_edges = []
    for i in range(num_units):
        G_unit = generate_3_regular_graph(vertices_per_unit)
        ep = remove_edge_and_get_endpoints(G_unit)
        removed_edges.append(ep)
        units.append(G_unit)

    G_union = nx.disjoint_union_all(units)
    offsets = [i * vertices_per_unit for i in range(num_units)]
    dangling = []
    for i in range(num_units):
        u, v = removed_edges[i]
        offset = offsets[i]
        dangling.append((u + offset, v + offset))

    for i in range(num_units):
        u_i = dangling[i][0]
        next_index = (i + 1) % num_units
        v_next = dangling[next_index][1]
        G_union.add_edge(u_i, v_next)

    return G_union

def generate_Erdo_Renyi_graph(num_qbts, edge_prob, seed=None):
    """Generates an Erdős-Rényi random graph."""
    G = nx.erdos_renyi_graph(num_qbts, edge_prob, seed=seed)
    return G

def edge_extract(G,Str="Blind"):
    assert Str == "Blind" or Str=="Naive",print("Str should be Naive or Blind")
    if Str == "Blind":
        return list(G.edges())
    else:
        communities = nx.community.greedy_modularity_communities(G, resolution=.8)
        community_lists = [list(c) for c in communities]
        mapping = {}
        new_label = 0
        for cluster_idx, cluster in enumerate(community_lists):
            for node in cluster:
                mapping[node] = new_label
                new_label += 1
        G_relabeled = nx.relabel_nodes(G, mapping)
        return list(G_relabeled.edges())

In [48]:
N=12
import numpy as np
from functools import reduce

# Pauli matrices
I = np.eye(2)
X = np.array([[0, 1],
              [1, 0]])
Z = np.array([[1, 0],
              [0, -1]])

def kron_n(ops):
    """Kronecker product de lista de operadores"""
    return reduce(np.kron, ops)

def build_ising_1D(N, J=1.0, h=1.0):
    """
    Hamiltoniano Ising cuántico 1D:
    H = -J sum Z_i Z_{i+1} - h sum X_i
    """
    H = np.zeros((2**N, 2**N), dtype=complex)

    # término ZZ
    for i in range(N - 1):
        ops = [I] * N
        ops[i] = Z
        ops[i + 1] = Z
        H += -J * kron_n(ops)

    # término campo transversal X
    for i in range(N):
        ops = [I] * N
        ops[i] = X
        H += -h * kron_n(ops)

    return H
    N = 6
H = build_ising_1D(N, J=1.0, h=1.0)



La función main hace:

-Simula un circuito cuántico (random o QAOA)

-Representarlo como red tensorial (MPS o TTN)

-Comprimirlo con DMRG

-Medir fidelidad del estado reconstruido

-Estudiar cómo cambia con bond dimension D

Responde a: ¿Puede una red tensorial representar un estado cuántico generado por un circuito? NO está resolviendo el Ising directamente, está aproximando un estado cuántico generado

In [49]:
from src.simulation import DMRG
from src.utils.TN_gen import D_tree, D_mps
from src.utils.graph_gen import * # graph types
from tqdm import tqdm
import numpy as np

def main():
    #Estás definiendo un sistema 1D de N qubits con interacción local, Ising/chain-like topology implícita
    no_qubits = 27                  # Total number of qubits
    qubit_graph = nearest_neighbour_edge_list(no_qubits)  # Qubit connectivity graph
    circuit_type = "random"         # Circuit type: "random" or "qaoa"
    Depth = 5                       # Circuit depth (number of layers)

    network_type = "tree"           # Network type: "tree" (TTN) or "mps"
    qubit_order = "Blind"           # TTN ordering: "Naive" or "Blind", "Blind" → orden natural, "Naive" → reordenamiento por comunidades
    network_structure = [1, 3, 9, 27]  # TTN structure (branching hierarchy), controla cómo se agrupan correlaciones

    compression_steps = 2           # Compression steps per depth
    no_sweeps = 2                   # Number of DMRG sweeps
    Dmax = [4, 8, 12]               # List of bond dimensions to test, D pequeño → mala aproximación, D grande → mejor pero más caro
    runs = 2                        # Number of independent runs per setup



    Fidelity_list = []

    print("Starting simulations...")

    for d_max in tqdm(Dmax, desc="D values"):

        if network_type == "tree":
            D = D_tree(network_structure,d_max)
        else:
            D = D_mps(no_qubits,d_max)
        Fidelity_run = 0
        for i in tqdm(range(runs), desc="Runs", leave=False): #reduce ruido estadístico
            edge_list = edge_extract(qubit_graph,qubit_order)
            fidelity_results = DMRG(  #Ejecuta DMRG, construir estado inicial (random MPS/TTN), aplicar circuito (random o QAOA), comprimirlo, hacer sweeps variacionales, optimizar tensores locales
                compression_steps=compression_steps,
                depth=Depth,
                no_sweeps=no_sweeps,
                no_qubits=no_qubits,
                D=D,
                network_structure=network_structure,
                full_edge_list=edge_list,
                network_type=network_type,
                circuit_type=circuit_type,
                run=i
            )
            print(f"Run {i+1} with Dmax={d_max} complete. Final Fidelity: {fidelity_results}")
            Fidelity_run += fidelity_results
        Fidelity_list.append(Fidelity_run/runs) #NO mide energía directamente, sino la fidelidad

    print("Bond Dimension, Fidelity")
    data = np.column_stack(( Dmax, Fidelity_list))
    print(data)

if __name__ == "__main__":
    main()

Starting simulations...


Runs:   0%|          | 0/2 [00:00<?, ?it/s]

 Total Compression Step = 10 and network = tree and nodes: [1, 3, 9, 27]
Number of 2 qubit gates: 13
Memory =  1120
Compression step = 1 and Depth = 1; Fidelity = 1.0000000000000022; sweep time = 6.596360683441162
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 1; Fidelity = 1.0000000000000038; sweep time = 3.8152172565460205
Number of 2 qubit gates: 13
Memory =  1120
Compression step = 1 and Depth = 2; Fidelity = 0.5256333348450366; sweep time = 4.283510208129883
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 2; Fidelity = 0.31265983177974455; sweep time = 5.11611533164978
Number of 2 qubit gates: 13
Memory =  1120
Compression step = 1 and Depth = 3; Fidelity = 0.12979128130641862; sweep time = 3.7952237129211426
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 3; Fidelity = 0.04672859480121767; sweep time = 3.733654260635376
Number of 2 qubit gates: 13
Memory =  1120
Compression step = 1 and Depth = 4; Fidelity = 0.018515104105020594; sweep t


Runs:  50%|█████     | 1/2 [00:49<00:49, 49.41s/it]

Run 1 with Dmax=4 complete. Final Fidelity: 0.0009586246719529911
 Total Compression Step = 10 and network = tree and nodes: [1, 3, 9, 27]
Number of 2 qubit gates: 13
Memory =  1120
Compression step = 1 and Depth = 1; Fidelity = 1.000000000000002; sweep time = 6.076367378234863
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 1; Fidelity = 1.0000000000000049; sweep time = 5.539025783538818
Number of 2 qubit gates: 13
Memory =  1120
Compression step = 1 and Depth = 2; Fidelity = 0.5307611780804615; sweep time = 4.185732841491699
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 2; Fidelity = 0.32793075233547453; sweep time = 4.490580797195435
Number of 2 qubit gates: 13
Memory =  1120
Compression step = 1 and Depth = 3; Fidelity = 0.11569406021262515; sweep time = 6.467330455780029
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 3; Fidelity = 0.035252367062804; sweep time = 4.787418842315674
Number of 2 qubit gates: 13
Memory =  1120
Compression st

...

 Total Compression Step = 10 and network = tree and nodes: [1, 3, 9, 27]
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 1; Fidelity = 1.0000000000000027; sweep time = 6.954883098602295
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 1; Fidelity = 1.000000000000005; sweep time = 5.389093399047852
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 2; Fidelity = 1.0000000000000056; sweep time = 6.6558756828308105
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 2; Fidelity = 0.9409542083445386; sweep time = 5.181596755981445
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 3; Fidelity = 0.7582652937397758; sweep time = 9.02318525314331
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 3; Fidelity = 0.6748799878730669; sweep time = 7.1307690143585205
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 4; Fidelity = 0.5085410146467673; sweep tim


Runs:  50%|█████     | 1/2 [01:08<01:08, 68.13s/it]

 Total Compression Step = 10 and network = tree and nodes: [1, 3, 9, 27]
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 1; Fidelity = 1.0000000000000004; sweep time = 6.743987560272217
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 1; Fidelity = 1.000000000000001; sweep time = 5.218976974487305
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 2; Fidelity = 1.0000000000000024; sweep time = 6.698590278625488
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 2; Fidelity = 0.9969885527774756; sweep time = 5.190382719039917
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 3; Fidelity = 0.8899683469763839; sweep time = 5.899236679077148
Number of 2 qubit gates: 13
Compression step = 2  and Depth = 3; Fidelity = 0.7826477901611327; sweep time = 6.814095735549927
Number of 2 qubit gates: 13
Memory =  20736
Compression step = 1 and Depth = 4; Fidelity = 0.6198713749933774; sweep time


D values: 100%|██████████| 3/3 [06:02<00:00, 120.83s/it]

Compression step = 2  and Depth = 5; Fidelity = 0.26572069576179613; sweep time = 6.158667087554932
Run 2 with Dmax=12 complete. Final Fidelity: 0.26572069576179613
Bond Dimension, Fidelity
[[4.00000000e+00 9.79910788e-04]
 [8.00000000e+00 1.63548403e-01]
 [1.20000000e+01 2.49110413e-01]]
